In [18]:
# ---------------- Imports ----------------
import os
import json
import sys

from pathlib import Path

import yaml
import pandas as pd
import inflect
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm



In [19]:
# ---------------- Config ----------------
with open("../../config/config.yaml", "r") as f:
    config = yaml.safe_load(f)


proj_dir = config["paths"]["proj_store"]
data_path = os.path.join(proj_dir, "data")


output_path = os.path.join(proj_dir, "evaluation", "dataset-analysis")

output_values = f"{output_path}/values"
os.makedirs(output_values, exist_ok=True)
output_tables = f"{output_path}/tables"
os.makedirs(output_tables, exist_ok=True)

main_folder_path = f"{data_path}/yield-v1"

fine_tuned_folder_path = f"{data_path}/yield-v1-finetuning"

comparison_data_path = f"{data_path}/supplementary/comparison-datasets.csv"


### Base dataset

In [20]:

def count_json_files_turns_tokens(folder_path):
    folder_data = []
    domain_data = []

    # Traverse each of the split directories
    for split in ['train', 'dev', 'test']:
        split_path = Path(folder_path) / split

        for json_file in split_path.rglob("*.json"):
            try:
                with json_file.open("r", encoding="utf-8") as f:
                    dialogues = json.load(f)

                if not isinstance(dialogues, list):
                    print(f"Skipping non-list file: {json_file}")
                    continue

                for i, dialogue in enumerate(dialogues):
                    dialogue_id = dialogue.get("dialogue_id", f"{json_file.stem}-{i}")
                    domain = dialogue.get("domain", "Unknown")
                    subdomain = dialogue.get("broad_source", "Unknown")
                    turns = dialogue.get("turns", [])

                    num_turns = len(turns)
                    num_tokens = sum(len(turn.get("utterance", "").split()) for turn in turns)

                    # Add row for folder-level aggregation
                    folder_data.append({
                        "Split": split,
                        "Dialogues": 1,
                        "Turns": num_turns,
                        "Tokens": num_tokens
                    })

                    # Add row for domain-level aggregation
                    domain_data.append({
                        "Domain": domain,
                        "Broad Source": subdomain,
                        "JSON File": f"{json_file.name}#{i}",
                        "Turns": num_turns,
                        "Tokens": num_tokens,
                        "Split": split
                    })

            except Exception as e:
                print(f"Error processing {json_file}: {e}")

    return folder_data, domain_data


# --------- Run analysis and build dataframes ---------

folder_data, domain_records = count_json_files_turns_tokens(main_folder_path)

# --- Folder-level aggregation (by split) ---
folder_df = pd.DataFrame(folder_data)

if folder_df.empty:
    print("No folder-level data found.")
else:
    split_aggregated_df = folder_df.groupby("Split").agg(
        Dialogues=("Dialogues", "sum"),
        Turns=("Turns", "sum"),
        Tokens=("Tokens", "sum")
    ).reset_index()

    # Add total row
    total_row = {
        "Split": "Total",
        "Dialogues": split_aggregated_df["Dialogues"].sum(),
        "Turns": split_aggregated_df["Turns"].sum(),
        "Tokens": split_aggregated_df["Tokens"].sum()
    }
    split_aggregated_df = pd.concat([split_aggregated_df, pd.DataFrame([total_row])], ignore_index=True)

# --- Domain-level aggregation ---
domain_df = pd.DataFrame(domain_records)

if domain_df.empty:
    print("No valid dialogue records found.")
else:
    aggregated_domain_df = domain_df.groupby(["Domain", "Broad Source"]).agg(
        Dialogues=("JSON File", "count"),
        Turns=("Turns", "sum"),
        Tokens=("Tokens", "sum")
    ).reset_index()

    # Totals row
    totals_row = {
        "Domain": "Total",
        "Broad Source": "",
        "Dialogues": aggregated_domain_df["Dialogues"].sum(),
        "Turns": aggregated_domain_df["Turns"].sum(),
        "Tokens": aggregated_domain_df["Tokens"].sum()
    }
    aggregated_domain_df = pd.concat([aggregated_domain_df, pd.DataFrame([totals_row])], ignore_index=True)

    # Formatting
    aggregated_domain_df['Domain'] = aggregated_domain_df['Domain'].str.replace('_', ' ').str.title()
    aggregated_domain_df['Broad Source'] = aggregated_domain_df['Broad Source'].str.replace('_', ' ').str.title()
    
    replacements = {
    "Harvard Dataverse": "Harvard",
    "Other Collections": "Other",
    "Voa News": "VOA",
    "Wikinews": "Wikinews",
    "Oyez Supreme Court": "Oyez",
    "Jfk Library": "JFK",
    "Jsc Oral History": "JSC",
    "Nara": "NARA",
    }

    aggregated_domain_df["Broad Source"] = aggregated_domain_df["Broad Source"].replace(replacements)


    replacements = {
    "Academic Interviews": "Academic",
    "Journalistic Interviews": "Journalistic",
    "Judicial Dialogue": "Judicial",
    "Oral History": "Oral",
          	
    }

    aggregated_domain_df["Domain"] = aggregated_domain_df["Domain"].replace(replacements)



In [21]:
aggregated_domain_df["Turns"] = (aggregated_domain_df["Turns"] / 1000).round(1)
aggregated_domain_df["Tokens"] = (aggregated_domain_df["Tokens"] / 1000).round(1)

display(aggregated_domain_df)



,Domain,Broad Source,Dialogues,Turns,Tokens
0,Academic,Harvard,136,26.3,654.4
1,Academic,Other,12,1.2,56.7
2,Journalistic Investigations,VOA,73,2.2,150.2
3,Journalistic Investigations,Wikinews,56,3.2,172.5
4,Judicial Proceedings,Oyez,621,162.9,7655.5
5,Oral,JFK,270,83.0,3056.4
6,Oral,JSC,1012,102.0,13626.4
7,Oral,NARA,101,9.5,876.9
8,Total,,2281,390.2,26249.0


In [22]:
# Remove the "Broad Source" column
domain_only_df = domain_df.drop(columns=["Broad Source"])

# Aggregate by Domain
domain_summary_df = domain_only_df.groupby("Domain").agg(
    Dialogues=("JSON File", "count"),
    Turns=("Turns", "sum"),
    Tokens=("Tokens", "sum")
).reset_index()

# Add total row
domain_total_row = {
    "Domain": "Total",
    "Dialogues": domain_summary_df["Dialogues"].sum(),
    "Turns": domain_summary_df["Turns"].sum(),
    "Tokens": domain_summary_df["Tokens"].sum()
}
domain_summary_df = pd.concat([domain_summary_df, pd.DataFrame([domain_total_row])], ignore_index=True)

# Format domain names
domain_summary_df["Domain"] = domain_summary_df["Domain"].str.replace("_", " ").str.title()


In [23]:
domain_summary_df

,Domain,Dialogues,Turns,Tokens
0,Academic Interviews,148,27476,711139
1,Journalistic Investigations,129,5342,322768
2,Judicial Proceedings,621,162894,7655484
3,Oral History,1383,194493,17559623
4,Total,2281,390205,26249014


In [24]:
split_order = ["train", "dev", "test", "Total"]

split_aggregated_df["Split"] = pd.Categorical(
    split_aggregated_df["Split"],
    categories=split_order,
    ordered=True
)

split_aggregated_df = split_aggregated_df.sort_values("Split").reset_index(drop=True)

# Display the folder-level DataFrame
print("Folder-level Aggregation:")
display(split_aggregated_df.head(30))



Folder-level Aggregation:


,Split,Dialogues,Turns,Tokens
0,train,1824,309495,21041238
1,dev,228,41330,2584725
2,test,229,39380,2623051
3,Total,2281,390205,26249014


### Comparison

In [25]:
# Import comparison data
comparison_data = pd.read_csv(comparison_data_path)

# Rename the columns to have the desired names
comparison_data = comparison_data.rename(columns={
    "dataset": "Dataset",
    "domains": "Domains",
    "dialogues": "Dialogues",
    "turns": "Turns",
    "tokens": "Tokens",
    "avg_turns_per_dialogue": "Avg. Turns Per Dialogue",
    "avg_tokens_per_turn": "Avg. Tokens Per Turn",
})


display(comparison_data)


,Dataset,Domains,Dialogues,Turns,Tokens,Avg. Turns Per Dialogue,Avg. Tokens Per Turn
0,DSTC2,1,1612,23354,199431,14.49,8.54
1,MultiWOZ,7,8438,113556,1490615,13.46,13.13
2,SGD,16,16142,329964,3217149,20.44,9.75


In [26]:
comparison_df = domain_summary_df[domain_summary_df['Domain'] == 'Total']

# Calculate
comparison_df = comparison_df.copy()
comparison_df["Avg. Turns Per Dialogue"] = comparison_df["Turns"] / comparison_df["Dialogues"]
comparison_df["Avg. Tokens Per Turn"] = comparison_df["Tokens"] / comparison_df["Turns"]


# Rename the columns to have the desired names
comparison_df = comparison_df.rename(columns={
    "Domain": "Dataset",
})


comparison_df['Dataset'] = comparison_df['Dataset'].str.replace('Total', 'YIELD')


comparison_df = pd.concat([comparison_data, comparison_df], ignore_index=True)

unique_domains_count = domain_summary_df[domain_summary_df['Domain'] != "Total"]['Domain'].nunique()

comparison_df.loc[comparison_df['Dataset'] == "YIELD", 'Domains'] = unique_domains_count


display(comparison_df)

,Dataset,Domains,Dialogues,Turns,Tokens,Avg. Turns Per Dialogue,Avg. Tokens Per Turn
0,DSTC2,1.0,1612,23354,199431,14.490000,8.540000
1,MultiWOZ,7.0,8438,113556,1490615,13.460000,13.130000
2,SGD,16.0,16142,329964,3217149,20.440000,9.750000
3,YIELD,4.0,2281,390205,26249014,171.067514,67.269804


### Save individual values

In [27]:
# save values

# Initialize inflect engine
p = inflect.engine()
				


search_value = 'YIELD'

columns_to_get = {
    'Domains': f"{output_values}/total_domains.txt",
    'Dialogues': f"{output_values}/total_dialogues.txt",
    'Turns': f"{output_values}/total_turns.txt",
    'Tokens': f"{output_values}/total_tokens.txt",
    'Avg. Turns Per Dialogue': f"{output_values}/avg_turns_per_dialogue.txt",
    'Avg. Tokens Per Turn': f"{output_values}/avg_tokens_per_turn.txt"
}

# Custom decimal formatting rules
decimal_format = {
    'Domains': 0,
    'Dialogues': 0,
    'Turns': 0,
    'Tokens': 0,
    'Avg. Turns Per Dialogue': 2,
    'Avg. Tokens Per Turn': 2
}

for column, file_path in columns_to_get.items():
    matching_value = comparison_df.loc[comparison_df['Dataset'] == search_value, column].values[0]
    decimals = decimal_format.get(column, 2)  # Default to 2 if not specified
    formatted_value = f"{matching_value:,.{decimals}f}"

    print(f"{column}: {formatted_value}")

    with open(file_path, "w") as file:
        file.write(formatted_value)



Domains: 4
Dialogues: 2,281
Turns: 390,205
Tokens: 26,249,014
Avg. Turns Per Dialogue: 171.07
Avg. Tokens Per Turn: 67.27


In [28]:
# Initialize inflect engine
p = inflect.engine()

# Get number of unique domains (make sure it's not counting YIELD/Total)
unique_counts = domain_summary_df[domain_summary_df['Domain'] != "Total"]['Domain'].nunique()

# Convert number to words (e.g., "twenty-one")
unique_counts_word = p.number_to_words(unique_counts)

print(unique_counts_word)

# Save to file
with open(f"{output_values}/total_domains_word.txt", "w") as f:
    f.write(unique_counts_word)


four


In [29]:

def get_folder_size(folder_path):
    total_size = 0
    for dirpath, dirnames, filenames in os.walk(folder_path):
        for file in filenames:
            file_path = os.path.join(dirpath, file)
            if os.path.isfile(file_path):
                total_size += os.path.getsize(file_path)
    return total_size

def format_size(size_in_bytes):
    sizes = ["Bytes", "KB", "MB", "GB", "TB"]
    i = 0
    while size_in_bytes >= 1024 and i < len(sizes) - 1:
        size_in_bytes /= 1024
        i += 1
    return f"{size_in_bytes:.2f} {sizes[i]}"

base_size_in_bytes = get_folder_size(main_folder_path)
base_formatted_size = format_size(base_size_in_bytes)

print(f"Folder Size: {base_formatted_size}")

# Save to a text file
with open(f"{output_values}/size_base.txt", "w") as file:
    file.write(str(base_formatted_size))




Folder Size: 201.82 MB


### Save tables

In [30]:
aggregated_domain_df.to_csv(f"{output_tables}/aggregated_domain_df.csv", index=False)


In [31]:
domain_summary_df.to_csv(f"{output_tables}/domain_summary_df.csv", index=False)


In [32]:
split_aggregated_df.to_csv(f"{output_tables}/split_aggregated_df.csv", index=False)


In [33]:
comparison_df.to_csv(f"{output_tables}/comparison_table.csv", index=False)


### Fine tuning dataset

In [34]:
def count_and_save_jsonl_blocks(base_folder, output_folder):
    stats = []

    for split in ['train', 'dev', 'test']:
        split_path = Path(base_folder) / split
        total_blocks = 0

        if not split_path.exists():
            print(f"{split_path} does not exist.")
            continue

        for jsonl_file in split_path.rglob("*.jsonl"):
            try:
                with jsonl_file.open("r", encoding="utf-8") as f:
                    total_blocks += sum(1 for _ in f)
            except Exception as e:
                print(f"Error reading {jsonl_file}: {e}")

        stats.append({
            "Split": split,
            "Blocks": total_blocks
        })

        # Save to individual .txt file
        output_file = Path(output_folder) / f"total_blocks_{split}.txt"
        with output_file.open("w") as f:
            f.write(f"{total_blocks:,}")

        print(f"{split.capitalize()}: {total_blocks} blocks written to {output_file}")

    # Create DataFrame
    df = pd.DataFrame(stats)

    # Add total row
    total_row = {
        "Split": "Total",
        "Blocks": df["Blocks"].sum()
    }
    df = pd.concat([df, pd.DataFrame([total_row])], ignore_index=True)

    return df

# usage
blocks_df = count_and_save_jsonl_blocks(
    base_folder=fine_tuned_folder_path,
    output_folder=f"{output_values}"
)

# display the DataFrame
display(blocks_df)



Train: 94488 blocks written to /data/yield-v1/evaluation/dataset-analysis/values/total_blocks_train.txt
Dev: 12973 blocks written to /data/yield-v1/evaluation/dataset-analysis/values/total_blocks_dev.txt
Test: 12283 blocks written to /data/yield-v1/evaluation/dataset-analysis/values/total_blocks_test.txt


,Split,Blocks
0,train,94488
1,dev,12973
2,test,12283
3,Total,119744
